# A primer the pipeline writes for itself

`sysml/aql_examples.md` is the only thing this project tells AQLizer about the graph,
and it is hand-written. Every paragraph in it was learned the same way: ask a
question, watch the generated AQL come back with no rows, work out why, write the
reason down. That works, and it does not scale past the models it was written for. A
SysML corpus imported next week has different attribute names, different requirement
identifiers, different snapshots -- and nobody to spend a week asking it questions.

`sysml/pipeline/examples.py` is the alternative: a build step that writes the primer
from the graph it will be used on, with a stronger model than the one answering
questions. It gets two things and nothing else.

**A fixed prompt** holding what is true of *any* graph this pipeline builds, because
the pipeline builds it that way -- names upper-cased, types lower-cased and closed,
`attributes` a map of `{value, unit}`, `files` and `models` lists, `stated` on what
the lexer read, containment as `owns` + `typedby`, time-varying values on snapshots.
That is most of what the hand-written file knows, stated without reference to a drone
or a Saturn V.

**A survey** of the live graph: seventeen AQL probes reporting which entity types
actually occur, which attribute names exist and in what units, real short names, real
snapshot names, how contested names were disambiguated, which relations were read and
which inferred. This is the corpus-specific half, and it is read rather than written.

Then every ```aql block in what comes back is parsed, run and counted, and anything
that fails goes back to the model with its error for one repair round. A worked
example that does not run is worse than none: the chain copies its shape.

So the question here is not "can a model write documentation". It is: **on a SysML
model nobody has looked at, how much of the hand-written primer's value survives being
generated?** What follows is a proxy for that, and a pessimistic one -- this corpus is
the one the hand-written file was tuned against, which makes it the easiest possible
test for the incumbent and the hardest for its replacement.

---

## Build both

The generated one is written here, live. The hand-written one already exists, so
"building" it means putting it through the same check: parse every query in it, run
it, count the rows.

In [1]:
import json
import logging
import time

from sysml import config, nl
from sysml.pipeline import examples

logging.disable(logging.INFO)
db = config.db()

LOG = config.OUT / "bespoke-aql-examples.log"


def note(line=""):
    print(line)
    with LOG.open("a", encoding="utf-8") as fh:
        fh.write(f"{line}\n")

In [2]:
built = examples.build(db, rounds=1)

print(f"{built['model']} wrote {len(built['markdown']):,} characters "
      f"from a {len(built['survey']):,}-character survey of the graph\n")
for i, line in enumerate(built["rounds"]):
    print(f"  {'generated' if not i else f'repair {i}':>9}  {line}")

gpt-5.5 wrote 17,436 characters from a 16,179-character survey of the graph

  generated  24 queries, 23 run, 1 do not parse, 22 return rows, 1 return nothing
   repair 1  24 queries, 24 run, 0 do not parse, 24 return rows, 0 return nothing


The repair round is not decoration. The first draft is written from the survey alone,
and the model has no way to know that a filter it invented matches nothing until
something runs it.

Running it is also the step that has to be gated. Checking a generated primer means
executing AQL a model wrote, against the graph it was describing, so `examples.check`
refuses anything matching `config.MUTATION` before it reaches the database -- the same
gate `nl` puts in front of a generated *answer*, for the same reason. Neither draft
here contained a write, but nothing about the way they are produced rules one out.

In [3]:
PRIMERS = {"none": None,
           "generated": config.AQL_EXAMPLES_GENERATED,
           "hand-written": config.AQL_EXAMPLES}

for name, path in PRIMERS.items():
    if path is None:
        continue
    text = path.read_text(encoding="utf-8")
    results = examples.check(db, text)
    binds = sum(1 for r in results if r["binds"])
    print(f"{name:>12}  {len(text):>6,} chars  {text.count(chr(10)):>4} lines")
    print(f"{'':>12}  {examples.report(results, examples.invented(db, text))}")
    print(f"{'':>12}  {binds} of {len(results)} queries use a bind parameter\n")

   generated  17,436 chars   587 lines
              24 queries, 24 run, 0 do not parse, 24 return rows, 0 return nothing
              0 of 24 queries use a bind parameter



hand-written  23,762 chars   535 lines
              26 queries, 26 run, 0 do not parse, 23 return rows, 3 return nothing
              9 of 26 queries use a bind parameter



Two numbers there are not what they look like.

The hand-written file's queries that return nothing are mostly its bind parameters:
`@name`, `@moment`, `@start` are placeholders, and the check fills them with values
sampled off the graph rather than the value the example was written for. The generated
file was *told* to prefer literals from the survey, so its queries are self-contained
and the check can really run them -- which is why they all return rows, and also why
the file is more tied to this corpus than it looks.

---

## The questions

Ten questions over three tiers, each with an answer the graph really gives. The answers
are computed first, by AQL written and checked by hand, and the tests below are scored
against those rows -- so nothing here is graded against an opinion, and a rebuild that
changes the graph changes the expected answers with it.

  **lookup**   one row, one field: read a value, say where it came from.
  **schema**   the question names something the graph spells differently -- an
               identifier that lives in `short_name`, a noun that is not an
               `entity_type`, a plural that is not one element.
  **shape**    the answer is a traversal, an aggregate or an anti-join, and getting the
               shape wrong returns rows that look fine and are wrong.

Each is asked three ways: with no examples at all (what the deployed service does),
with the generated primer, and with the hand-written one.

In [4]:
TRUTH = {
 "drone-requirements": '''
    FOR e IN sysml_Entities
      FILTER e.entity_type == 'requirement' AND 'DroneModelLogical' IN e.models
      COLLECT WITH COUNT INTO n RETURN n''',

 "dry-mass": '''
    FOR e IN sysml_Entities FILTER e.entity_name == 'S-IC'
      RETURN {value: e.attributes.dryMass.value, unit: e.attributes.dryMass.unit,
              file: e.source_file, line: e.source_line}''',

 "failure-rate": '''
    FOR e IN sysml_Entities FILTER e.attributes.failureRate.value != null
      SORT e.attributes.failureRate.value DESC
      RETURN {element: e.entity_name, rate: e.attributes.failureRate.value,
              unit: e.attributes.failureRate.unit}''',

 "short-name": '''
    FOR e IN sysml_Entities FILTER e.short_name == 'HLR-R049'
      RETURN {name: e.entity_name, file: e.source_file, line: e.source_line,
              refined_by: (FOR v, r IN 1..1 ANY e sysml_Relations
                             FILTER r.relationship_type == 'refines' RETURN v.entity_name)}''',

 "batteries": '''
    FOR e IN sysml_Entities FILTER CONTAINS(e.entity_name, 'BATTERY')
      RETURN {name: e.entity_name, type: e.entity_type}''',

 "rollup": '''
    FOR e IN sysml_Entities FILTER e.entity_name == 'SATURNV'
      LET parts = (FOR c, edge IN 1..6 OUTBOUND e sysml_Relations
                     FILTER edge.relationship_type IN ['owns', 'typedby']
                     FILTER c.attributes.dryMass.value != null
                     RETURN DISTINCT {name: c.entity_name, kg: c.attributes.dryMass.value})
      RETURN {total: SUM(parts[*].kg), contributors: parts}''',

 "anti-join": '''
    LET reqs = (FOR e IN sysml_Entities
                  FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
                  RETURN e)
    RETURN {total: LENGTH(reqs), unsatisfied: LENGTH(
      FOR e IN reqs FILTER LENGTH(FOR r IN sysml_Relations
        FILTER r._to == e._id AND r.relationship_type == 'satisfies' RETURN 1) == 0
      RETURN 1)}''',

 "provenance": '''
    FOR r IN sysml_Relations FILTER r.type == 'RELATED_TO'
      LET from = DOCUMENT(r._from) FILTER from != null
      FOR m IN from.models
        COLLECT model = m, read = r.stated == true WITH COUNT INTO n
        RETURN {model, source: read ? 'read' : 'inferred', n}''',

 "moment": '''
    FOR o IN sysml_Entities FILTER o.entity_name == 'ATLOI'
      FOR v, e, p IN 1..5 OUTBOUND o sysml_Relations
        FILTER p.edges[*].relationship_type ALL IN ['owns', 'redefines', 'typedby']
        FILTER LENGTH(ATTRIBUTES(v.attributes)) > 0
        RETURN DISTINCT {element: v.entity_name, values: v.attributes}''',

 # The pair is stored with the lexicographically smaller id first, so which end
 # is the Apollo one is not fixed -- sort the two ends by model rather than by
 # edge direction.
 "analogy": '''
    FOR r IN sysml_Relations
      FILTER r.type == 'SIMILAR_TO' AND r.analogy_role == 'part'
      LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
      LET a_apollo = 'apollo-11-sysml-v2' IN a.models
      FILTER a_apollo != ('apollo-11-sysml-v2' IN b.models)
      SORT r.cosine DESC LIMIT 5
      RETURN {drone: a_apollo ? b.entity_name : a.entity_name,
              apollo: a_apollo ? a.entity_name : b.entity_name,
              cosine: ROUND(r.cosine * 100) / 100}''',
}

truth = {name: list(db.aql.execute(query)) for name, query in TRUTH.items()}
for name, rows in truth.items():
    print(f"{name:>18}  {json.dumps(rows, default=str)[:130]}")

drone-requirements  [33]
          dry-mass  [{"value": 137000, "unit": "kg", "file": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml", "line": 217}]
      failure-rate  [{"element": "SATURNVINSTRUMENTUNIT", "rate": 5e-06, "unit": "1/h"}, {"element": "TECHNICALCOMPONENTSPACKAGE_LUNARMODULEDESCENTSTA
        short-name  [{"name": "LUNARLANDERASCENTREADINESSTIME", "file": "apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml", "line": 44
         batteries  [{"name": "DRONEBATTERY", "type": "package"}, {"name": "BATTERY", "type": "part"}, {"name": "DRONE_BATTERY", "type": "part"}, {"na
            rollup  [{"total": 188650, "contributors": [{"name": "SATURNVINSTRUMENTUNIT", "kg": 1950}, {"name": "S-IVB", "kg": 13500}, {"name": "S-II"
         anti-join  [{"total": 610, "unsatisfied": 352}]
        provenance  [{"model": "apollo-11-sysml-v2", "source": "inferred", "n": 564}, {"model": "apollo-11-sysml-v2", "source": "read", "n": 2938}, {"
            moment  [{"

Every expectation below is taken from those rows. `needs` is what the answer must
contain, `any_of` is a choice, and `some` is "at least this many of these" for the
questions whose answer is a set rather than a value.

In [5]:
apollo = {row["source"]: row["n"] for row in truth["provenance"]
          if row["model"] == "apollo-11-sysml-v2"}
mass = truth["dry-mass"][0]
declared_in = mass["file"].split("/")[-1]
requirement = truth["short-name"][0]

TESTS = [
 dict(id="drone-requirements", tier="lookup",
      q="How many requirements are there in the DroneModelLogical model?",
      needs=[str(truth["drone-requirements"][0])]),

 dict(id="dry-mass", tier="lookup",
      q="What is the dry mass of the S-IC stage, and which file and line declares it?",
      needs=[str(mass["value"])], any_of=[declared_in, str(mass["line"])]),

 dict(id="failure-rate", tier="schema",
      q="Which components have a failure rate? Rank them highest first and give the unit.",
      needs=[truth["failure-rate"][0]["element"]],
      any_of=[truth["failure-rate"][0]["unit"]]),

 dict(id="short-name", tier="schema",
      q="What is the requirement HLR-R049 -- what is it called, where is it declared, "
        "and what refines it?",
      needs=[requirement["name"]],
      any_of=[requirement["file"].split("/")[-1], str(requirement["line"])]),

 dict(id="batteries", tier="schema",
      q="Tell me about batteries in these models -- what is there?",
      some=(4, [row["name"] for row in truth["batteries"]])),

 dict(id="rollup", tier="shape",
      q="Add up the dry mass of everything the Saturn V is made of, and show each "
        "contributor with its own mass.",
      needs=[str(truth["rollup"][0]["total"])]),

 dict(id="anti-join", tier="shape",
      q="How many Apollo requirements have nothing satisfying them?",
      needs=[str(truth["anti-join"][0]["unsatisfied"])]),

 dict(id="provenance", tier="shape",
      q="Break the relations down by model and by whether they were read from the "
        "syntax or inferred by the LLM.",
      needs=[str(apollo["read"]), str(apollo["inferred"])]),

 dict(id="moment", tier="shape",
      q="What condition is the spacecraft in at lunar orbit insertion, and what "
        "values does the model give for that moment?",
      needs=[str(truth["moment"][0]["values"]["fuelRemaining"]["value"])],
      any_of=["Burning", "SPS"]),

 dict(id="analogy", tier="shape",
      q="Which drone parts have an Apollo counterpart, and how close is the match?",
      some=(2, [row["apollo"] for row in truth["analogy"]])),
]

# A question whose answer is empty is not a test of anything. The analogy layer is
# the one part of the graph a `load` on its own does not rebuild, so it is checked
# rather than assumed.
TESTS = [t for t in TESTS if truth[t["id"]]]
if len(TESTS) < 10:
    dropped = [name for name, rows in truth.items() if not rows]
    print(f"dropped {dropped}: the graph has no rows for it")
    print()

for test in TESTS:
    wanted = test.get("needs", []) + list(test.get("any_of", []))
    if "some" in test:
        wanted = [f"{test['some'][0]} of {len(test['some'][1])}"]
    print(f"{test['tier']:>6}  {test['id']:<19} expects {wanted}")

lookup  drone-requirements  expects ['33']
lookup  dry-mass            expects ['137000', 'TechnicalComponentsPackage.sysml', '217']
schema  failure-rate        expects ['SATURNVINSTRUMENTUNIT', '1/h']
schema  short-name          expects ['LUNARLANDERASCENTREADINESSTIME', 'MissionRequirementsPackage.sysml', '448']
schema  batteries           expects ['4 of 21']
 shape  rollup              expects ['188650']
 shape  anti-join           expects ['352']
 shape  provenance          expects ['2938', '564']
 shape  moment              expects ['18400', 'Burning', 'SPS']
 shape  analogy             expects ['2 of 5']


In [6]:
def flat(text):
    "Comparable text: no thousands separators, no case, no markdown punctuation."
    return " ".join("".join(c for c in text if c not in ",_`*'\"").split()).lower()


def ask(question, primer):
    if primer is None:                      # the deployed service, told nothing
        return nl.instance().ask(question, primed=False)
    return nl.instance(primer).ask(question)


def score(answer, test):
    blob = flat(answer.answer + " " + json.dumps(answer.rows, default=str))
    missed = [n for n in test.get("needs", []) if flat(n) not in blob]
    if test.get("any_of") and not any(flat(n) in blob for n in test["any_of"]):
        missed.append(" or ".join(test["any_of"]))
    if "some" in test:
        wanted, names = test["some"]
        found = sum(1 for n in set(names) if flat(n) in blob)
        if found < wanted:
            missed.append(f"named {found} of them, wanted {wanted}")
    return not missed, missed


RESULTS, ANSWERS = [], {}

for test in TESTS:
    note(f"\n{test['tier']:>6}  {test['q']}")
    for primer_name, primer in PRIMERS.items():
        started = time.time()
        answer = ask(test["q"], primer)
        ok, missed = score(answer, test)
        ANSWERS[(test["id"], primer_name)] = answer
        RESULTS.append(dict(id=test["id"], tier=test["tier"], primer=primer_name,
                            ok=ok, missed=missed, rows=len(answer.rows),
                            error=answer.error, seconds=round(time.time() - started, 1)))
        note(f"        {primer_name:>12}  {'PASS' if ok else 'fail':<5} "
             f"{len(answer.rows):>3} rows  {RESULTS[-1]['seconds']:>5}s  "
             + ("" if ok else f"missing {missed}")
             + (f"  [{answer.error[:60]}]" if answer.error else ""))


lookup  How many requirements are there in the DroneModelLogical model?


                none  fail    1 rows    3.8s  missing ['33']


           generated  PASS    1 rows    2.0s  


        hand-written  PASS    1 rows    2.6s  

lookup  What is the dry mass of the S-IC stage, and which file and line declares it?


                none  fail    0 rows    3.4s  missing ['137000', 'TechnicalComponentsPackage.sysml or 217']


...

           generated  PASS    4 rows    3.7s  


        hand-written  PASS    4 rows    4.2s  

 shape  Which drone parts have an Apollo counterpart, and how close is the match?


                none  fail   41 rows    7.3s  missing ['named 1 of them, wanted 2']


           generated  PASS    1 rows    4.4s  


        hand-written  PASS    6 rows    3.8s  


---

## The scores

`empty` is the failure this project cares most about: the query ran, returned nothing,
and the chain wrote a fluent sentence over zero rows. `wrong` means it returned rows
and the answer is not what the graph says.

In [7]:
def table(rows, columns):
    header = {c: c for c in columns}
    widths = [max(len(str(r[c])) for r in rows + [header]) for c in columns]
    print("  ".join(c.ljust(w) for c, w in zip(columns, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows:
        print("  ".join(str(row[c]).ljust(w) for c, w in zip(columns, widths)))


grid = []
for test in TESTS:
    row = {"question": test["id"], "tier": test["tier"]}
    for primer_name in PRIMERS:
        r = next(x for x in RESULTS if x["id"] == test["id"] and x["primer"] == primer_name)
        row[primer_name] = ("pass" if r["ok"] else "ERROR" if r["error"]
                            else "empty" if not r["rows"] else "wrong")
    grid.append(row)

table(grid, ["question", "tier", *PRIMERS])
print()
for primer_name in PRIMERS:
    mine = [r for r in RESULTS if r["primer"] == primer_name]
    print(f"{primer_name:>12}  {sum(1 for r in mine if r['ok'])}/{len(mine)} correct   "
          f"{sum(1 for r in mine if not r['rows'] and not r['error'])} returned no rows   "
          f"{sum(1 for r in mine if r['error'])} failed to run   "
          f"{sum(r['seconds'] for r in mine) / len(mine):.0f}s per question")

print()
for tier in ("lookup", "schema", "shape"):
    line = f"{tier:>8}  "
    for primer_name in PRIMERS:
        mine = [r for r in RESULTS if r["primer"] == primer_name and r["tier"] == tier]
        line += f"{primer_name} {sum(1 for r in mine if r['ok'])}/{len(mine)}   "
    print(line)

question            tier    none   generated  hand-written
------------------  ------  -----  ---------  ------------
drone-requirements  lookup  wrong  pass       pass        
dry-mass            lookup  empty  wrong      pass        
failure-rate        schema  ERROR  pass       pass        
short-name          schema  empty  pass       pass        
batteries           schema  empty  pass       pass        
rollup              shape   empty  pass       pass        
anti-join           shape   wrong  pass       pass        
provenance          shape   ERROR  ERROR      wrong       
moment              shape   empty  pass       pass        
analogy             shape   wrong  pass       pass        

        none  0/10 correct   5 returned no rows   2 failed to run   5s per question
   generated  8/10 correct   0 returned no rows   1 failed to run   5s per question
hand-written  9/10 correct   0 returned no rows   0 failed to run   4s per question

  lookup  none 0/2   generated 1/2   h

### Is that difference noise?

The chain writes a fresh query every time it is asked, so one run is one sample. The
same battery runs twice more below -- `P` is a pass, `.` a miss, one character per
run including the one above.

In [8]:
REPEATS = 2
repeat = {(t["id"], name): [] for t in TESTS for name in PRIMERS if name}

for _ in range(REPEATS):
    for test in TESTS:
        for primer_name, primer in PRIMERS.items():
            ok, _ = score(ask(test["q"], primer), test)
            repeat[(test["id"], primer_name)].append(ok)

runs = 1 + REPEATS
for test in TESTS:
    line = f"{test['id']:>19}  "
    for primer_name in PRIMERS:
        first = next(r["ok"] for r in RESULTS
                     if r["id"] == test["id"] and r["primer"] == primer_name)
        marks = "".join("P" if ok else "." for ok in [first] + repeat[(test["id"], primer_name)])
        line += f"{primer_name} {marks}   "
    note(line)

note("")
for primer_name in PRIMERS:
    passed = sum(1 for r in RESULTS if r["primer"] == primer_name and r["ok"])         + sum(sum(repeat[(t["id"], primer_name)]) for t in TESTS)
    note(f"{primer_name:>12}  {passed}/{len(TESTS) * runs} over {runs} runs")

 drone-requirements  none ...   generated PPP   hand-written PP.   
           dry-mass  none ...   generated ...   hand-written PPP   
       failure-rate  none ...   generated PPP   hand-written PPP   
         short-name  none ...   generated PPP   hand-written PP.   
          batteries  none ...   generated PPP   hand-written PPP   
             rollup  none ...   generated P..   hand-written PPP   
          anti-join  none ...   generated PPP   hand-written PP.   
         provenance  none ...   generated ...   hand-written .PP   
             moment  none ...   generated PPP   hand-written PPP   
            analogy  none .P.   generated PP.   hand-written PPP   

        none  1/30 over 3 runs
   generated  21/30 over 3 runs
hand-written  26/30 over 3 runs


---

## Where they differ, in the AQL

A score says which one was right. The queries say why, and the why is the part that
transfers to a corpus this notebook has never seen.

In [9]:
def compare(test_id, *primers):
    test = next(t for t in TESTS if t["id"] == test_id)
    print(f"Q  {test['q']}\n")
    for primer_name in primers or PRIMERS:
        answer = ANSWERS[(test_id, primer_name)]
        result = next(r for r in RESULTS
                      if r["id"] == test_id and r["primer"] == primer_name)
        print(f"--- {primer_name}  ({'correct' if result['ok'] else 'wrong'}, "
              f"{len(answer.rows)} rows)")
        for line in (answer.aql or "(no query)").strip().splitlines():
            print(f"    {line}")
        print(f"    A  {answer.answer[:400]}\n")


compare("rollup")

Q  Add up the dry mass of everything the Saturn V is made of, and show each contributor with its own mass.

--- none  (wrong, 0 rows)
    WITH sysml_Chunks, sysml_Communities, sysml_Documents, sysml_Entities
    FOR entity IN sysml_Entities
      FILTER entity.entity_name == "Saturn V" && entity.entity_type == "component"
      FOR v, e, p IN 1..1 OUTBOUND entity sysml_Relations
        FILTER e.type == "PART_OF"
        LET mass = (
          FOR m IN sysml_Entities
            FILTER m._id == v._id && m.entity_type == "attribute" && CONTAINS(m.entity_name, "mass")
            RETURN m
        )
        RETURN { component: v.entity_name, mass: LENGTH(mass) > 0 ? mass[0] : null }
    A  The query returned no rows. It seems there is no available data regarding the dry mass of the components that make up the Saturn V in the database, according to the criteria specified.

--- generated  (correct, 1 rows)
    WITH sysml_Communities, sysml_Documents, sysml_Chunks, sysml_Entities
    LET sat

In [10]:
compare("short-name")

Q  What is the requirement HLR-R049 -- what is it called, where is it declared, and what refines it?

--- none  (wrong, 0 rows)
    WITH sysml_Entities, sysml_Documents, sysml_Chunks, sysml_Communities
    
    FOR vertex, edge 
    IN OUTBOUND SHORTEST_PATH "sysml_Entities/requirement_hlr-r049" TO "ANY_VERTEX"
    GRAPH "sysml_kg" 
    
    FILTER edge.type == "DECLARED_IN" OR edge.type == "REFINES"
    
    COLLECT entityName = vertex.entity_name, fileName = vertex.file_name
    WITH COUNT INTO countRefines
    
    RETURN {
      entityName,
      fileName,
      countRefines
    }
    A  The query returned no information regarding the requirement "hlr-r049." It did not find details about what the requirement is called, where it is declared, or what refines it.

--- generated  (correct, 1 rows)
    LET r = FIRST(
      FOR e IN sysml_Entities
        FILTER e.short_name == "HLR-R049"
        RETURN e
    )
    
    LET refining_entities = (
      FOR v, e IN 1..1 OUTBOUND r sysml_Re

In [11]:
compare("moment")

Q  What condition is the spacecraft in at lunar orbit insertion, and what values does the model give for that moment?

--- none  (wrong, 0 rows)
    WITH sysml_Chunks, sysml_Communities, sysml_Documents, sysml_Entities
    FOR doc IN sysml_Documents
      FILTER CONTAINS(doc.content, "lunar orbit insertion")
      FOR chunk IN sysml_Chunks
        FILTER chunk._id == doc._id
        RETURN {
          condition: doc.content,
          modelValues: chunk.content
        }
    A  The query returned no rows, indicating that the current database does not provide specific information about the condition of the spacecraft at lunar orbit insertion or the model values associated with that moment.

--- generated  (correct, 4 rows)
    WITH sysml_Entities, sysml_Relations
    LET t = FIRST(
      FOR e IN sysml_Entities
        FILTER e.entity_name == "LUNARORBITINSERTION"
        FILTER e.entity_type == "timeslice"
        RETURN e
    )
    
    FOR v, e, p IN 1..6 OUTBOUND t sysml_Relations
 

In [12]:
compare("provenance", "generated", "hand-written")

Q  Break the relations down by model and by whether they were read from the syntax or inferred by the LLM.

--- generated  (wrong, 0 rows)
    (no query)
    A  

--- hand-written  (wrong, 3 rows)
    WITH sysml_Entities, sysml_Relations
    FOR r IN sysml_Relations
      FILTER r.type == "RELATED_TO"
      LET fromEntity = DOCUMENT(r._from)
      FILTER fromEntity != null
      COLLECT model = fromEntity.models[**] WITH COUNT INTO relationCount
      RETURN {
        model: model,
        source: "read from the syntax",
        relations: relationCount
      }
    A  The query results indicate a breakdown of relations by model, specifically focusing on those entities where relations were "read from the syntax." The models identified are: 

1. **Apollo-11-sysml-v2** with 3,502 relations.
2. **Drone_BaseArchitecture** with 10 relations.
3. **DroneModelLogical** with 182 relations.

Each model's relation count reflects connections that were directly read from the



---

## What this says about a model nobody has looked at

Over three runs of ten questions:

| primer | correct | what it costs |
|---|---|---|
| none -- the deployed service | 1/30 | nothing |
| generated, from the graph | 21/30 | one strong-model call per import, about three minutes |
| hand-written | 26/30 | a week of asking questions, once, per corpus |

The first row is the one that sets the scale. Without a primer this graph is close to
unusable for analytical questions: the chain invents an `entity_type` of `component`,
compares an upper-case name against `"Saturn V"`, walks `PART_OF` looking for
containment, and reports the empty result as an absence in the model. That is the
starting position for every SysML corpus this project has not seen.

**What the generated primer gets, it gets for the same reason the hand-written one
does.** Every question in the schema tier -- the identifier that lives in
`short_name`, the noun that is not a type, the plural that is not one element, the
values that hang under a snapshot -- it answers three times out of three, with queries
that are recognisably the same shape. Those traps are properties of how this pipeline
writes a graph, they were stated in the prompt without naming a drone or a Saturn V,
and the survey supplied the corpus-specific half. That is the part that transfers to a
model nobody has looked at, and it is most of the file.

**What it misses, it misses consistently, and in one direction.** Its three weak
questions are all arithmetic or edge discipline:

- *dry mass of S-IC* (0/3) -- it looked for `S-IC_STAGE`. The prompt tells it names
  are prefixed with their owner where several declarations share one; it applied that
  to a name that was never contested. The hand-written file, which shows the bare form
  in a worked example, does not.
- *provenance breakdown* (0/3) -- it counts every edge in the collection, so the
  importer's own `MENTIONED_IN` and `IN_COMMUNITY` edges land in the "inferred by the
  LLM" bucket and the number comes out five times too big. `FILTER type ==
  'RELATED_TO'` is one line, and its absence is invisible: the query runs, the rows
  look plausible, and the answer is wrong.
- *mass rollup* (1/3) -- the traversal is right every time, including the path filter
  and the `owns`/`typedby` pair, but two runs in three it returned one row per
  contributor and never summed. The question said "add up".

Each of those is a rule, not a fact about drones: *count in AQL*, *the edge collection
holds more than SysML relations*, *read the element's own attribute before traversing
from it*. They are missing from the generated file because the prompt does not say
them, and they are missing from the prompt because the hand-written file learned them
from watching answers go wrong rather than from looking at the graph. Moving them into
`GUIDE` costs three sentences and would carry to every future corpus -- which is the
concrete next step this run argues for.

**The honest caveats.**

- This corpus is where the hand-written file grew up. On a genuinely new SysML model
  it would start from whatever generalises, and the generated one would start from a
  survey of the actual graph. The 21-vs-26 gap is the pessimistic reading.
- Three runs is a small sample and the chain is stochastic: the hand-written primer
  also lost three questions once each, and one question (`provenance`) beat both
  primers in the run above.
- The scoring is a substring test against values computed from the graph. It rewards
  an answer that contains the right number and cannot tell a well-reasoned answer from
  a lucky one.
- The generated file is graded on the graph it was written from. That is not leakage
  here -- it is the intended use, one primer per import -- but it does mean this
  measures *fit to a corpus*, not generalisation across corpora.

**So: is it good enough for new SysML models we import?** For everything except
aggregation, yes -- it recovers most of the distance between an unprimed chain and a
week of hand-tuning, and it does it in three minutes with no human in the loop. For
questions that end in a total, a count or a provenance split, it is not there yet, and
those are exactly the questions a systems engineer asks. The fix is not more
generation; it is three more rules in a prompt that is already doing the work.
